# IPL Data Exploration
**Purpose:** Parse, clean and validate ball-by-ball IPL data from cricsheet.org  
**Data:** 1169 IPL matches (2008–2025)  
**Output:** `../data/processed/ipl_deliveries.csv`

## 1. Imports & Setup

In [2]:
import json
import os
import pandas as pd
from tqdm import tqdm

IPL_PATH = "../data/raw/ipl"

files = [f for f in os.listdir(IPL_PATH) if f.endswith('.json')]
print(f"IPL match files found: {len(files)}")

IPL match files found: 1169


## 2. Parser

In [3]:
def parse_match(filepath):
    """Parse a single cricsheet JSON match file into a list of delivery rows."""
    with open(filepath) as f:
        match = json.load(f)

    info = match['info']
    rows = []

    for innings_num, innings in enumerate(match['innings']):
        team_batting = innings['team']

        for over_data in innings['overs']:
            over = over_data['over']

            for ball_num, delivery in enumerate(over_data['deliveries']):
                row = {
                    # Match level
                    'match_id':     os.path.basename(filepath).replace('.json', ''),
                    'date':         info['dates'][0],
                    'venue':        info.get('venue'),
                    'city':         info.get('city'),
                    'season':       info.get('season'),
                    'match_number': info.get('event', {}).get('match_number'),
                    'winner':       info.get('outcome', {}).get('winner'),
                    # Innings level
                    'innings':      innings_num + 1,
                    'batting_team': team_batting,
                    'bowling_team': [t for t in info['teams'] if t != team_batting][0],
                    # Ball level
                    'over':         over,
                    'ball':         ball_num + 1,
                    'batter':       delivery['batter'],
                    'bowler':       delivery['bowler'],
                    'non_striker':  delivery['non_striker'],
                    # Runs
                    'runs_batter':  delivery['runs']['batter'],
                    'runs_extras':  delivery['runs']['extras'],
                    'runs_total':   delivery['runs']['total'],
                    # Extras
                    'wide':         delivery.get('extras', {}).get('wides', 0),
                    'no_ball':      delivery.get('extras', {}).get('noballs', 0),
                    'bye':          delivery.get('extras', {}).get('byes', 0),
                    'leg_bye':      delivery.get('extras', {}).get('legbyes', 0),
                    # Wicket
                    'wicket':       1 if 'wickets' in delivery else 0,
                    'wicket_kind':  delivery.get('wickets', [{}])[0].get('kind'),
                    'player_out':   delivery.get('wickets', [{}])[0].get('player_out'),
                }
                rows.append(row)

    return rows

## 3. Cleaning Functions

In [4]:
def clean_season(season):
    """
    Standardise season to calendar year integer.
      '2007/08' -> 2008  (take second part)
      '2009/10' -> 2010  (take second part)
      '2020/21' -> 2020  (special case: Covid IPL played in 2020)
    """
    season = str(season)
    if '/' in season:
        if season == '2020/21':
            return 2020
        second = season.split('/')[1]
        if len(second) == 2:
            prefix = season.split('/')[0][:2]
            return int(prefix + second)
        return int(second)
    return int(season)


TEAM_MAPPING = {
    'Delhi Daredevils':            'Delhi Capitals',
    'Kings XI Punjab':             'Punjab Kings',
    'Rising Pune Supergiant':      'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
}

## 4. Parse All Matches

In [5]:
all_rows = []

for filename in tqdm(files):
    filepath = os.path.join(IPL_PATH, filename)
    try:
        all_rows.extend(parse_match(filepath))
    except Exception as e:
        print(f"Error parsing {filename}: {e}")

df_ipl = pd.DataFrame(all_rows)

# Apply cleaning
df_ipl['season']       = df_ipl['season'].apply(clean_season)
df_ipl['batting_team'] = df_ipl['batting_team'].replace(TEAM_MAPPING)
df_ipl['bowling_team'] = df_ipl['bowling_team'].replace(TEAM_MAPPING)

print(f"Shape: {df_ipl.shape}")
print(f"Seasons: {sorted(df_ipl['season'].unique())}")
print(f"\nMatches per season:")
print(df_ipl.groupby('season')['match_id'].nunique().to_string())

100%|██████████| 1169/1169 [00:01<00:00, 830.03it/s]


Shape: (278205, 25)
Seasons: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Matches per season:
season
2008    58
2009    57
2010    60
2011    73
2012    74
2013    76
2014    60
2015    59
2016    60
2017    59
2018    60
2019    60
2020    60
2021    60
2022    74
2023    74
2024    71
2025    74


## 5. Save Processed Data

In [6]:
output_path = "../data/processed/ipl_deliveries.csv"
df_ipl.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")

Saved: ../data/processed/ipl_deliveries.csv
File size: 47.5 MB


## 6. Validation
### 6a. Orange Cap — Top Run Scorer per Season
Excludes wides. Validated 100% against official IPL records.

In [7]:
run_scorers = (df_ipl[df_ipl['wide'] == 0]
               .groupby(['season', 'batter'])['runs_batter']
               .sum()
               .reset_index()
               .sort_values(['season', 'runs_batter'], ascending=[True, False])
               .groupby('season')
               .first()
               .reset_index())

print("=== TOP RUN SCORER PER SEASON ===")
print(run_scorers[['season', 'batter', 'runs_batter']].to_string(index=False))

=== TOP RUN SCORER PER SEASON ===
 season          batter  runs_batter
   2008        SE Marsh          616
   2009       ML Hayden          572
   2010    SR Tendulkar          618
   2011        CH Gayle          608
   2012        CH Gayle          733
   2013      MEK Hussey          733
   2014      RV Uthappa          660
   2015       DA Warner          562
   2016         V Kohli          973
   2017       DA Warner          641
   2018   KS Williamson          735
   2019       DA Warner          692
   2020        KL Rahul          676
   2021      RD Gaikwad          635
   2022      JC Buttler          863
   2023    Shubman Gill          890
   2024         V Kohli          741
   2025 B Sai Sudharsan          759


### 6b. Purple Cap — Top Wicket Taker per Season
Excludes non-bowler dismissals (run out, retired hurt, retired out, obstructing).  
Validated against official IPL records — 17/18 correct. Rabada 2020 shows 32 vs official 30 (minor cricsheet discrepancy).

In [8]:
NON_BOWLER_DISMISSALS = ['run out', 'retired hurt', 'retired out', 'obstructing the field']

wicket_takers = (df_ipl[
        (df_ipl['wicket'] == 1) &
        (~df_ipl['wicket_kind'].isin(NON_BOWLER_DISMISSALS))
    ]
    .groupby(['season', 'bowler'])['wicket']
    .sum()
    .reset_index()
    .sort_values(['season', 'wicket'], ascending=[True, False])
    .groupby('season')
    .first()
    .reset_index())

print("=== TOP WICKET TAKER PER SEASON ===")
print(wicket_takers[['season', 'bowler', 'wicket']].to_string(index=False))

=== TOP WICKET TAKER PER SEASON ===
 season            bowler  wicket
   2008     Sohail Tanvir      22
   2009          RP Singh      23
   2010           PP Ojha      21
   2011        SL Malinga      28
   2012          M Morkel      25
   2013          DJ Bravo      32
   2014         MM Sharma      23
   2015          DJ Bravo      26
   2016           B Kumar      23
   2017           B Kumar      26
   2018            AJ Tye      24
   2019       Imran Tahir      26
   2020          K Rabada      32
   2021          HV Patel      32
   2022         YS Chahal      27
   2023    Mohammed Shami      28
   2024          HV Patel      24
   2025 M Prasidh Krishna      25


In [ ]:
#checking descrepancy of Rabada's wickets in 2020
rabada_2020_detail = df_ipl[
    (df_ipl['bowler'] == 'K Rabada') &
    (df_ipl['season'] == 2020) &
    (df_ipl['wicket'] == 1) &
    (~df_ipl['wicket_kind'].isin(['run out', 'retired hurt', 'retired out']))
].groupby('match_id').agg(
    date=('date', 'first'),
    wickets=('wicket', 'sum'),
    players_out=('player_out', lambda x: ', '.join(x.dropna()))
).sort_values('date').reset_index()

print(rabada_2020_detail.to_string(index=False))
print(f"\nTotal wickets: {rabada_2020_detail['wickets'].sum()}")
# Remove phantom innings (T20 only has 2 innings)
before = len(df_ipl)
df_ipl = df_ipl[df_ipl['innings'] <= 2]
after = len(df_ipl)
print(f"Removed {before - after} rows with innings > 2")print(f"Total matches: {len(rabada_2020_detail)}")

match_id       date  wickets                                     players_out
 1216493 2020-09-20        4       GJ Maxwell, K Gowtham, KL Rahul, N Pooran
 1216539 2020-09-25        3               F du Plessis, MS Dhoni, RA Jadeja
 1216532 2020-09-29        2                      JM Bairstow, KS Williamson
 1216515 2020-10-03        1                                      AD Russell
 1216519 2020-10-05        4     V Kohli, Washington Sundar, S Dube, I Udana
 1216500 2020-10-09        3                  JC Archer, R Tewatia, VR Aaron
 1216529 2020-10-11        2                          SA Yadav, Ishan Kishan
 1216543 2020-10-14        1                                       JC Archer
 1216509 2020-10-17        1                                    F du Plessis
 1216546 2020-10-20        2                            N Pooran, GJ Maxwell
 1216497 2020-10-24        2                           KD Karthik, SP Narine
 1216505 2020-11-02        2                             JR Philippe, S Dube

In [ ]:
# Remove phantom innings (T20 only has 2 innings)
before = len(df_ipl)
df_ipl = df_ipl[df_ipl['innings'] <= 2]
after = len(df_ipl)
print(f"Removed {before - after} rows with innings > 2")

rabada_2020_check = df_ipl[
    (df_ipl['bowler'] == 'K Rabada') &
    (df_ipl['season'] == 2020) &
    (df_ipl['wicket'] == 1) &
    (~df_ipl['wicket_kind'].isin(['run out', 'retired hurt', 'retired out']))
].groupby('match_id').agg(
    date=('date', 'first'),
    wickets=('wicket', 'sum'),
    players_out=('player_out', lambda x: ', '.join(x.dropna()))
).sort_values('date').reset_index()

print(rabada_2020_check.to_string(index=False))
print(f"\nTotal wickets: {rabada_2020_check['wickets'].sum()}")

match_id       date  wickets                                     players_out
 1216493 2020-09-20        2                           GJ Maxwell, K Gowtham
 1216539 2020-09-25        3               F du Plessis, MS Dhoni, RA Jadeja
 1216532 2020-09-29        2                      JM Bairstow, KS Williamson
 1216515 2020-10-03        1                                      AD Russell
 1216519 2020-10-05        4     V Kohli, Washington Sundar, S Dube, I Udana
 1216500 2020-10-09        3                  JC Archer, R Tewatia, VR Aaron
 1216529 2020-10-11        2                          SA Yadav, Ishan Kishan
 1216543 2020-10-14        1                                       JC Archer
 1216509 2020-10-17        1                                    F du Plessis
 1216546 2020-10-20        2                            N Pooran, GJ Maxwell
 1216497 2020-10-24        2                           KD Karthik, SP Narine
 1216505 2020-11-02        2                             JR Philippe, S Dube

In [13]:
df_ipl.to_csv("../data/processed/ipl_deliveries.csv", index=False)
print(f"Saved. Shape: {df_ipl.shape}")

Saved. Shape: (278034, 25)


In [14]:
# Remove phantom innings (T20 only has 2 innings — cricsheet occasional data issue)
df_ipl = df_ipl[df_ipl['innings'] <= 2]